# Week 2 – Data Collection, Cleaning, and Preprocessing for Logistics Analysis

**Student:** Isai Abinaya S  
**Organization / Internship Provider:** YuvalIntern  
**Role:** Logistics Data Analyst Intern  

## Objective
This notebook implements the Week 2 preprocessing workflow for a last-mile e-commerce logistics scenario. It uses the Brazilian E-Commerce Public Dataset by Olist and demonstrates data collection simulation, data inspection, cleaning, missing-value treatment, duplicate handling, validation, outlier detection, normalization/standardization, aggregation, feature engineering, and quality checks.

> **Important:** The notebook is an implementation template. Exact row counts and numerical outputs depend on the Olist CSV files available in the local `data/` folder. No fabricated results are used.


## 1. Logistics Scenario

The scenario focuses on **last-mile e-commerce delivery performance**. The objective is to prepare reliable order-level data for later analysis of delivery time, delays, freight cost, customer satisfaction, and other logistics KPIs.

### Main preprocessing goals
- Collect and load the required logistics datasets.
- Understand the structure and data types.
- Identify missing values and duplicates.
- Detect invalid values and logical inconsistencies.
- Detect and treat extreme numeric values using the IQR method where appropriate.
- Standardize selected numeric variables when required for machine-learning workflows.
- Aggregate one-to-many order-item data to an order level.
- Create delivery and delay features for future analysis.
- Validate the cleaned dataset before saving it.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)
    
print('Libraries imported successfully.')

## 2. Dataset and File Structure

The Olist dataset contains multiple related CSV files. For this preprocessing workflow, the main files are:

| File | Purpose |
|---|---|
| `olist_orders_dataset.csv` | Order dates, delivery dates, and order status |
| `olist_order_items_dataset.csv` | Products, sellers, prices, and freight values |
| `olist_order_reviews_dataset.csv` | Customer review scores and review information |
| `olist_customers_dataset.csv` | Customer identifiers and locations |
| `olist_sellers_dataset.csv` | Seller identifiers and locations |
| `olist_products_dataset.csv` | Product information |
| `olist_geolocation_dataset.csv` | Geolocation information |

Place the downloaded CSV files inside a local `data/` directory before running the loading cells.

In [ ]:
# Project paths
DATA_DIR = Path('data')
    
required_files = [
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_products_dataset.csv',
    'olist_geolocation_dataset.csv'
]

missing_files = [f for f in required_files if not (DATA_DIR / f).exists()]
    
if missing_files:
    print('The following files are missing:')
    for f in missing_files:
        print('-', f)
    print('\nDownload the Olist CSV files and place them inside the data/ folder.')
else:
    print('All required files are available.')

In [ ]:
# Load the main logistics datasets
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
    
print('Orders:', orders.shape)
print('Order items:', order_items.shape)
print('Reviews:', reviews.shape)
print('Customers:', customers.shape)
print('Sellers:', sellers.shape)
print('Products:', products.shape)

## 3. Initial Data Inspection

Before modifying the data, inspect dimensions, column names, data types, sample records, and summary statistics. This helps identify structural problems before preprocessing.

In [ ]:
print('Orders columns:')
print(orders.columns.tolist())

print('\nFirst five order records:')
display(orders.head())

print('\nData types:')
display(orders.dtypes)

In [ ]:
# Summary statistics for numeric columns
display(orders.describe(include='all').T)

## 4. Standardize Date and Time Columns

Logistics analysis depends heavily on accurate timestamps. Date columns are converted to pandas datetime format. Invalid date strings are converted to `NaT`, which can then be identified during quality checks.

In [ ]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')

display(orders[date_columns].dtypes)

## 5. Missing-Value Analysis

Missing data can occur because a process was not completed, information was not recorded, or a field is not applicable. Missing-value treatment should be based on the meaning of each variable rather than applying one method to every column.

In [ ]:
def missing_summary(df):
    result = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_percent': df.isna().mean() * 100
    })
    return result.sort_values('missing_count', ascending=False)

orders_missing = missing_summary(orders)
display(orders_missing[orders_missing['missing_count'] > 0])

### Missing-value treatment strategy

- Date fields are generally not filled with arbitrary dates because doing so can create false delivery information.
- Optional fields may remain missing when the absence itself has meaning.
- Numeric variables used in machine-learning models can be imputed when appropriate, using a documented strategy such as median imputation.
- For delivery-time calculations, records without the necessary timestamps are excluded from that particular calculation rather than inventing dates.

## 6. Duplicate Detection and Handling

Duplicate records can inflate counts and distort logistics KPIs. Exact duplicates are checked first. Key-based duplication is also examined because some tables naturally contain multiple rows per order.

In [ ]:
print('Exact duplicate rows in orders:', orders.duplicated().sum())
print('Duplicate order_id values:', orders['order_id'].duplicated().sum())

# Remove exact duplicates only when they exist.
orders = orders.drop_duplicates().copy()

print('Shape after exact-duplicate removal:', orders.shape)

## 7. Invalid Values and Logical Validation

Logistics data must satisfy logical relationships. For example, a delivered-customer timestamp should not normally occur before the purchase timestamp. The following checks identify suspicious records without silently deleting them.

In [ ]:
# Order status distribution
display(orders['order_status'].value_counts(dropna=False))

    # Check for negative delivery intervals where both dates exist
has_dates = orders['order_purchase_timestamp'].notna() & orders['order_delivered_customer_date'].notna()
negative_delivery = (
    orders.loc[has_dates, 'order_delivered_customer_date'] <
    orders.loc[has_dates, 'order_purchase_timestamp']
)

print('Records with delivery before purchase:', negative_delivery.sum())

In [ ]:
# Numeric validity checks in order_items
numeric_cols = ['price', 'freight_value']

for col in numeric_cols:
    if col in order_items.columns:
        order_items[col] = pd.to_numeric(order_items[col], errors='coerce')
        print(f'{col} negative values:', (order_items[col] < 0).sum())
        print(f'{col} missing values:', order_items[col].isna().sum())

## 8. Outlier Detection Using the IQR Method

Extreme values can strongly influence averages and machine-learning models. The Interquartile Range (IQR) method identifies observations outside the lower and upper fences.

Formula:

- `IQR = Q3 - Q1`
- `Lower bound = Q1 - 1.5 × IQR`
- `Upper bound = Q3 + 1.5 × IQR`

An outlier is not automatically an error. In logistics, an unusually expensive shipment or long delivery can be a genuine business event. Therefore, outliers should be investigated before removal or transformation.

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

for col in ['price', 'freight_value']:
    if col in order_items.columns:
        s = order_items[col].dropna()
        lower, upper = iqr_bounds(s)
        outlier_count = ((order_items[col] < lower) | (order_items[col] > upper)).sum()
    
        print(f'\n{col}')
        print('Lower bound:', lower)
        print('Upper bound:', upper)
        print('Potential outliers:', outlier_count)

## 9. Numeric Cleaning

For monetary variables, negative values are not expected in this dataset. Invalid negative values are treated as missing rather than being replaced with zero because zero would represent a valid economic value and could distort analysis.

In [ ]:
for col in ['price', 'freight_value']:
    if col in order_items.columns:
        order_items.loc[order_items[col] < 0, col] = np.nan

display(order_items[['price', 'freight_value']].describe())

## 10. Aggregate Order Items to Order Level

The order-items table has a one-to-many relationship with orders: one order may contain multiple items. For order-level logistics analysis, item-level price and freight values are aggregated by `order_id`.

In [ ]:
item_agg = (
    order_items.groupby('order_id', as_index=False)
    .agg(
        item_count=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight_value=('freight_value', 'sum')
    )
)

display(item_agg.head())
print('Aggregated order-level records:', len(item_agg))

## 11. Customer Review Aggregation

Review data may contain multiple review records associated with an order. The average review score is calculated at the order level for use as a customer-satisfaction indicator.

In [ ]:
review_agg = (
    reviews.groupby('order_id', as_index=False)
    .agg(
        review_score=('review_score', 'mean')
    )
)

display(review_agg.head())

## 12. Build the Clean Order-Level Dataset

The cleaned order-level dataset combines order information, aggregated item information, and aggregated review scores. A left join from the orders table preserves the order-level population while allowing missing review/item information to remain visible for quality assessment.

In [ ]:
clean_orders = orders.merge(item_agg, on='order_id', how='left')
clean_orders = clean_orders.merge(review_agg, on='order_id', how='left')

print('Clean order-level dataset shape:', clean_orders.shape)
display(clean_orders.head())

## 13. Feature Engineering for Delivery Analysis

Two important logistics features are created:

1. **Delivery time** – number of days from purchase to customer delivery.
2. **Delivery delay** – difference between actual customer delivery and estimated delivery.

These features support later KPI analysis and predictive modelling.

In [ ]:
clean_orders['delivery_time_days'] = (
    clean_orders['order_delivered_customer_date'] -
    clean_orders['order_purchase_timestamp']
).dt.total_seconds() / (24 * 60 * 60)

clean_orders['delivery_delay_days'] = (
    clean_orders['order_delivered_customer_date'] -
    clean_orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 60 * 60)

display(clean_orders[[
    'order_id', 'order_status', 'delivery_time_days',
    'delivery_delay_days'
]].head())

### Interpretation of delay

- A **positive** `delivery_delay_days` means the delivery occurred after the estimated date.
- A **negative** value means the order arrived before the estimated date.
- Missing values can occur for orders that were not delivered or lack required timestamps.

## 14. Standardization / Normalization

Scaling is useful when variables have very different numerical ranges, especially for distance-based models and many machine-learning algorithms. Here, `StandardScaler` is demonstrated on selected numeric features.

Standardization transforms a variable approximately as:

`z = (x - mean) / standard deviation`

The original business variables are retained, while scaled variables are stored with a `_scaled` suffix.

In [ ]:
scale_columns = [
    'total_price',
    'total_freight_value',
    'item_count',
    'delivery_time_days'
]

available_scale_columns = [c for c in scale_columns if c in clean_orders.columns]
scale_data = clean_orders[available_scale_columns].copy()

if available_scale_columns:
    # Fit only on complete rows for this demonstration.
    complete_mask = scale_data.notna().all(axis=1)
    scaler = StandardScaler()
    clean_orders.loc[complete_mask, [c + '_scaled' for c in available_scale_columns]] = (
        scaler.fit_transform(scale_data.loc[complete_mask, available_scale_columns])
    )

display(clean_orders.head())

## 15. Data Quality Validation Checklist

A final validation stage checks whether the preprocessing pipeline produced a usable dataset.

In [ ]:
validation = {
    'rows': len(clean_orders),
    'columns': clean_orders.shape[1],
    'duplicate_order_ids': clean_orders['order_id'].duplicated().sum(),
    'missing_order_ids': clean_orders['order_id'].isna().sum(),
    'negative_delivery_times': (clean_orders['delivery_time_days'] < 0).sum(),
    'negative_delivery_delays': (clean_orders['delivery_delay_days'] < 0).sum()
}

validation_df = pd.DataFrame(validation.items(), columns=['check', 'value'])
display(validation_df)

In [ ]:
# Additional missing-value report after preprocessing
final_missing = missing_summary(clean_orders)
display(final_missing.head(15))

## 16. Reusable Preprocessing Function

The following function summarizes the core order preprocessing steps so the workflow can be reused on another copy of the same dataset.

In [ ]:
def preprocess_orders(orders_df, items_df, reviews_df):
    """Prepare an order-level logistics dataset."""
    orders_clean = orders_df.copy()
    items_clean = items_df.copy()
    reviews_clean = reviews_df.copy()

    # Remove exact duplicate rows.
    orders_clean = orders_clean.drop_duplicates()

    # Convert timestamps.
    date_cols = [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    for col in date_cols:
        if col in orders_clean.columns:
            orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

    # Numeric conversion and invalid-value handling.
    for col in ['price', 'freight_value']:
        if col in items_clean.columns:
            items_clean[col] = pd.to_numeric(items_clean[col], errors='coerce')
            items_clean.loc[items_clean[col] < 0, col] = np.nan

    # Aggregate item-level data.
    item_summary = items_clean.groupby('order_id', as_index=False).agg(
        item_count=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight_value=('freight_value', 'sum')
    )

    # Aggregate reviews.
    review_summary = reviews_clean.groupby('order_id', as_index=False).agg(
        review_score=('review_score', 'mean')
    )

    # Merge to order level.
    result = orders_clean.merge(item_summary, on='order_id', how='left')
    result = result.merge(review_summary, on='order_id', how='left')

    # Delivery features.
    result['delivery_time_days'] = (
        result['order_delivered_customer_date'] -
        result['order_purchase_timestamp']
    ).dt.total_seconds() / 86400

    result['delivery_delay_days'] = (
        result['order_delivered_customer_date'] -
        result['order_estimated_delivery_date']
    ).dt.total_seconds() / 86400

    return result

print('Reusable preprocessing function created.')

## 17. Save the Prepared Dataset

The cleaned order-level dataset can be saved for later exploratory analysis and modelling. The output directory is created automatically.

In [ ]:
OUTPUT_DIR = Path('processed_data')
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / 'clean_orders_week2.csv'
clean_orders.to_csv(output_file, index=False)

print(f'Saved cleaned dataset to: {output_file}')

## 18. Reflection: Why Data Quality Matters in Logistics

Data quality directly affects the reliability of logistics KPIs and downstream models. Duplicate records can inflate order counts, missing timestamps can make delivery-time calculations incomplete, invalid numeric values can distort cost metrics, and unexamined outliers can influence averages and predictive models.

The preprocessing workflow therefore separates **data cleaning** from **business interpretation**. A suspicious value is identified first; its meaning is considered before deciding whether to remove, replace, transform, or retain it. This approach reduces the risk of introducing artificial patterns into later analysis.

The prepared order-level dataset provides a foundation for future work such as delivery-performance analysis, customer segmentation, prediction of delivery delays, freight-cost analysis, and logistics optimization.

## 19. Conclusion

This Week 2 notebook demonstrates a structured data-preprocessing pipeline for logistics analysis using Python and pandas. The workflow covers dataset loading, inspection, datetime conversion, missing-value analysis, duplicate handling, logical validation, outlier detection, numeric cleaning, order-level aggregation, feature engineering, standardization, and final validation.

The main outcome is a cleaner and more consistent order-level dataset that can be used for subsequent logistics analytics. The exact quality metrics and resulting row counts should be reported only after the notebook is executed against the actual Olist files.


## References

1. Olist Brazilian E-Commerce Public Dataset – Kaggle.
2. pandas documentation – data manipulation and preprocessing.
3. scikit-learn documentation – `StandardScaler` and preprocessing.
4. YuvalIntern Week 2 task requirements.


## Suggested GitHub Project Structure

```text
Logistics-Data-Analytics-Virtual-Internship/
│
├── reports/
│   ├── Week1_Strategic_Planning_Report.docx
│   └── Week2_Data_Collection_Cleaning_Preprocessing_Report.docx
│
├── notebooks/
│   └── Week2_Data_Collection_Cleaning_Preprocessing.ipynb
│
├── data/
│   └── Olist CSV files (optional/local)
│
└── README.md
```

**Recommended commit message:** `Add Week 2 data preprocessing report and notebook`